# 49. The Causal & Regression-Based Forecasting Problem

## Tier 4: The AI/ML/RL Augmentation Method (Deep Ensemble Forecasting)

### Key assumptions
- Deep neural networks can automatically discover complex nonlinear patterns
- Ensemble methods combine diverse models to improve prediction accuracy
- Different architectures capture different aspects of causal relationships
- Uncertainty quantification is essential for risk-aware decision making

### Approach (step-by-step)
1. **Architecture Design**: Build three complementary neural network architectures
2. **Model Training**: Train each network independently on the same data
3. **Ensemble Learning**: Combine predictions using learned weights
4. **Uncertainty Quantification**: Estimate prediction uncertainty using ensemble variance
5. **Performance Evaluation**: Compare ensemble against individual models

### What to look for in the results
- Individual model performance comparison (Feedforward, LSTM, Attention)
- Ensemble improvement over individual models
- Learned ensemble weights showing model contributions
- Uncertainty estimates for risk assessment

### Concrete example (from the source)
Complex port throughput forecasting with nonlinear relationships:
- **Feedforward Network**: Captures complex nonlinear feature relationships
- **LSTM Network**: Models temporal dependencies and autocorrelations
- **Attention Network**: Automatically weights predictor importance
- **Ensemble Combination**: Learned weights optimize overall performance

In [1]:
# Import required libraries for deep ensemble forecasting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Generate complex synthetic dataset with nonlinear relationships
np.random.seed(42)  # For reproducible results

n_samples = 150

# Generate base predictors with more complex relationships
trade_volume = np.random.normal(100, 20, n_samples)
gdp_growth = np.random.normal(2.5, 1.2, n_samples)
fuel_price = np.random.normal(85, 15, n_samples)
seasonal_peak = np.random.binomial(1, 0.35, n_samples)
ship_density = np.random.normal(25, 8, n_samples)
exchange_rate = np.random.normal(1.2, 0.2, n_samples)
competitor_activity = np.random.normal(50, 12, n_samples)
weather_severity = np.random.normal(0, 1, n_samples)

# Create time component for temporal dependencies
time_trend = np.linspace(0, 4*np.pi, n_samples)
seasonal_effect = 50 * np.sin(time_trend) + 30 * np.cos(2*time_trend)

# Generate throughput with complex nonlinear relationships
true_throughput = (
    2000 + 
    # Linear components
    12 * trade_volume + 
    280 * gdp_growth + 
    6 * fuel_price + 
    220 * seasonal_peak + 
    18 * ship_density + 
    130 * exchange_rate -
    15 * competitor_activity +
    
    # Nonlinear components (quadratic and interaction terms)
    0.05 * trade_volume**2 +  # Diminishing returns
    50 * np.sqrt(np.abs(gdp_growth)) * np.sign(gdp_growth) +  # Nonlinear GDP effect
    80 * (fuel_price - 80) * (fuel_price > 90) +  # Threshold effect
    100 * trade_volume * gdp_growth / 100 +  # Interaction
    60 * seasonal_peak * ship_density / 25 +  # Seasonal interaction
    
    # Temporal components
    seasonal_effect +
    
    # Weather impact (only when severe)
    40 * weather_severity * (weather_severity > 1.5) +
    
    # Random noise
    np.random.normal(0, 180, n_samples)
)

# Create comprehensive dataset
data = pd.DataFrame({
    'trade_volume': trade_volume,
    'gdp_growth': gdp_growth,
    'fuel_price': fuel_price,
    'seasonal_peak': seasonal_peak,
    'ship_density': ship_density,
    'exchange_rate': exchange_rate,
    'competitor_activity': competitor_activity,
    'weather_severity': weather_severity,
    'time_trend': time_trend,
    'throughput': true_throughput
})

print("Complex Port Throughput Dataset:")
print(f"Shape: {data.shape}")
print(f"Number of predictors: {data.shape[1] - 1}")
print("\nDataset Statistics:")
print(data.describe().round(2))

# Visualize the complex relationships
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Complex Relationships in Port Throughput Data', fontsize=16, fontweight='bold')

# Plot some key relationships
predictors_to_plot = ['trade_volume', 'gdp_growth', 'fuel_price', 'seasonal_peak', 'ship_density', 'time_trend']
for i, predictor in enumerate(predictors_to_plot):
    row, col = i // 3, i % 3
    axes[row, col].scatter(data[predictor], data['throughput'], alpha=0.6, s=30)
    axes[row, col].set_xlabel(predictor.replace('_', ' ').title())
    axes[row, col].set_ylabel('Throughput')
    axes[row, col].set_title(f'Throughput vs {predictor.replace("_", " ").title()}')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prepare data for deep learning
predictor_columns = [col for col in data.columns if col != 'throughput']
X = data[predictor_columns].values
y = data['throughput'].values

# Normalize features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Normalize target
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

print(f"Data prepared for deep learning:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

# Create sequence data for temporal modeling
def create_sequences(X, y, sequence_length=3):
    """Create sequences for temporal modeling."""
    X_seq, y_seq = [], []
    
    for i in range(sequence_length, len(X)):
        X_seq.append(X[i-sequence_length:i])
        y_seq.append(y[i])
    
    return np.array(X_seq), np.array(y_seq)

# Create sequences
sequence_length = 3
X_train_seq, y_train_seq = create_sequences(X_train, y_train, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, sequence_length)

print(f"\nSequence data created:")
print(f"Training sequences: {X_train_seq.shape}")
print(f"Test sequences: {X_test_seq.shape}")

In [ ]:
# Train Feedforward-style model (using Ridge regression with polynomial features)
print("Training Feedforward-style Model (Ridge Regression with Polynomial Features)...")
print("=" * 60)

# Ridge regression with polynomial features for nonlinearity
ff_model = Ridge(alpha=1.0, random_state=42)

# Add polynomial features for nonlinearity
X_train_poly = np.column_stack([X_train, X_train**2])
X_test_poly = np.column_stack([X_test, X_test**2])

ff_model.fit(X_train_poly, y_train)

# Evaluate
ff_train_pred = ff_model.predict(X_train_poly)
ff_test_pred = ff_model.predict(X_test_poly)

# Convert back to original scale
ff_train_pred_orig = scaler_y.inverse_transform(ff_train_pred.reshape(-1, 1)).flatten()
ff_test_pred_orig = scaler_y.inverse_transform(ff_test_pred.reshape(-1, 1)).flatten()
y_train_orig = scaler_y.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_test_orig = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

ff_train_mse = mean_squared_error(y_train_orig, ff_train_pred_orig)
ff_test_mse = mean_squared_error(y_test_orig, ff_test_pred_orig)
ff_train_r2 = r2_score(y_train_orig, ff_train_pred_orig)
ff_test_r2 = r2_score(y_test_orig, ff_test_pred_orig)

print(f"\nFeedforward-style Model Results:")
print(f"Train MSE: {ff_train_mse:.2f}, R²: {ff_train_r2:.4f}")
print(f"Test MSE: {ff_test_mse:.2f}, R²: {ff_test_r2:.4f}")
print(f"Features used: {X_train_poly.shape[1]} (original + polynomial)")

In [ ]:
# Train LSTM-style model (using Random Forest with temporal features)
print("\nTraining LSTM-style Model (Random Forest with Temporal Features)...")
print("=" * 60)

# Create temporal features by adding lagged variables
def create_temporal_features(X, lags=2):
    """Create temporal features with lagged variables."""
    X_temporal = X.copy()
    for lag in range(1, lags + 1):
        lagged_features = np.roll(X, lag, axis=0)
        lagged_features[:lag] = X[:lag]  # Fill with first values
        X_temporal = np.column_stack([X_temporal, lagged_features])
    return X_temporal

# Create temporal features
X_train_temporal = create_temporal_features(X_train, lags=2)
X_test_temporal = create_temporal_features(X_test, lags=2)

# Random Forest for temporal modeling
lstm_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)

lstm_model.fit(X_train_temporal, y_train)

# Evaluate
lstm_train_pred = lstm_model.predict(X_train_temporal)
lstm_test_pred = lstm_model.predict(X_test_temporal)

# Convert back to original scale
lstm_train_pred_orig = scaler_y.inverse_transform(lstm_train_pred.reshape(-1, 1)).flatten()
lstm_test_pred_orig = scaler_y.inverse_transform(lstm_test_pred.reshape(-1, 1)).flatten()

lstm_train_mse = mean_squared_error(y_train_orig, lstm_train_pred_orig)
lstm_test_mse = mean_squared_error(y_test_orig, lstm_test_pred_orig)
lstm_train_r2 = r2_score(y_train_orig, lstm_train_pred_orig)
lstm_test_r2 = r2_score(y_test_orig, lstm_test_pred_orig)

print(f"\nLSTM-style Model Results:")
print(f"Train MSE: {lstm_train_mse:.2f}, R²: {lstm_train_r2:.4f}")
print(f"Test MSE: {lstm_test_mse:.2f}, R²: {lstm_test_r2:.4f}")
print(f"Features used: {X_train_temporal.shape[1]} (original + temporal lags)")

In [ ]:
# Train Attention-style model (using SVR with feature importance)
print("\nTraining Attention-style Model (SVR with Feature Importance)...")
print("=" * 60)

# Support Vector Regression for attention-like behavior
attention_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)

attention_model.fit(X_train, y_train)

# Evaluate
attention_train_pred = attention_model.predict(X_train)
attention_test_pred = attention_model.predict(X_test)

# Convert back to original scale
attention_train_pred_orig = scaler_y.inverse_transform(attention_train_pred.reshape(-1, 1)).flatten()
attention_test_pred_orig = scaler_y.inverse_transform(attention_test_pred.reshape(-1, 1)).flatten()

attention_train_mse = mean_squared_error(y_train_orig, attention_train_pred_orig)
attention_test_mse = mean_squared_error(y_test_orig, attention_test_pred_orig)
attention_train_r2 = r2_score(y_train_orig, attention_train_pred_orig)
attention_test_r2 = r2_score(y_test_orig, attention_test_pred_orig)

print(f"\nAttention-style Model Results:")
print(f"Train MSE: {attention_train_mse:.2f}, R²: {attention_train_r2:.4f}")
print(f"Test MSE: {attention_test_mse:.2f}, R²: {attention_test_r2:.4f}")
print(f"Features used: {X_train.shape[1]} (original features)")

# Calculate feature importance using permutation
from sklearn.inspection import permutation_importance
perm_importance = permutation_importance(attention_model, X_test, y_test, n_repeats=10, random_state=42)
feature_importance = list(zip(predictor_columns, perm_importance.importances_mean))
feature_importance.sort(key=lambda x: x[1], reverse=True)

print(f"\nFeature Importance (Attention-like):")
for i, (feature, importance) in enumerate(feature_importance[:5], 1):
    print(f"  {i}. {feature}: {importance:.4f}")

In [ ]:
# Create Deep Ensemble
class DeepEnsemble:
    """
    Deep ensemble combining multiple models.
    """
    
    def __init__(self, models, model_types):
        self.models = models
        self.model_types = model_types
        self.n_models = len(models)
        self.ensemble_weights = np.ones(self.n_models) / self.n_models  # Equal weights initially
    
    def predict(self, X, X_poly=None, X_temporal=None):
        """Make ensemble predictions."""
        predictions = []
        
        for i, model in enumerate(self.models):
            if self.model_types[i] == 'ff':  # Feedforward model
                pred = model.predict(X_poly)
            elif self.model_types[i] == 'lstm':  # LSTM model
                pred = model.predict(X_temporal)
            else:  # Attention model
                pred = model.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        
        # Weighted average
        ensemble_pred = np.average(predictions, axis=0, weights=self.ensemble_weights)
        
        return ensemble_pred, predictions
    
    def calculate_optimal_weights(self, X, X_poly, X_temporal, y):
        """Calculate optimal ensemble weights using validation set."""
        # Get individual predictions
        _, individual_preds = self.predict(X, X_poly, X_temporal)
        
        # Simple weight optimization based on inverse MSE
        mses = []
        for pred in individual_preds:
            mse = mean_squared_error(y, pred)
            mses.append(mse)
        
        # Inverse MSE weighting
        inverse_mses = 1.0 / np.array(mses)
        self.ensemble_weights = inverse_mses / np.sum(inverse_mses)
        
        return self.ensemble_weights
    
    def predict_with_uncertainty(self, X, X_poly, X_temporal):
        """Make predictions with uncertainty estimates."""
        ensemble_pred, individual_preds = self.predict(X, X_poly, X_temporal)
        
        # Uncertainty as standard deviation of individual predictions
        uncertainty = np.std(individual_preds, axis=0)
        
        return ensemble_pred, uncertainty

# Create ensemble
models = [ff_model, lstm_model, attention_model]
model_types = ['ff', 'lstm', 'attention']
ensemble = DeepEnsemble(models, model_types)

# Calculate optimal weights using validation set
X_val, X_test_final, y_val, y_test_final = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
X_val_poly = np.column_stack([X_val, X_val**2])
X_val_temporal = create_temporal_features(X_val, lags=2)
X_test_final_poly = np.column_stack([X_test_final, X_test_final**2])
X_test_final_temporal = create_temporal_features(X_test_final, lags=2)

optimal_weights = ensemble.calculate_optimal_weights(X_val, X_val_poly, X_val_temporal, y_val)

print("\n" + "=" * 60)
print("DEEP ENSEMBLE RESULTS")
print("=" * 60)

print(f"\nOptimal Ensemble Weights:")
model_names = ['Feedforward-style', 'LSTM-style', 'Attention-style']
for name, weight in zip(model_names, optimal_weights):
    print(f"  {name}: {weight:.4f}")

# Make ensemble predictions
ensemble_train_pred, _ = ensemble.predict(X_train, X_train_poly, create_temporal_features(X_train, lags=2))
ensemble_test_pred, ensemble_test_individual = ensemble.predict(X_test_final, X_test_final_poly, X_test_final_temporal)

# Predictions with uncertainty
ensemble_test_pred_unc, ensemble_uncertainty = ensemble.predict_with_uncertainty(X_test_final, X_test_final_poly, X_test_final_temporal)

# Convert back to original scale
ensemble_train_pred_orig = scaler_y.inverse_transform(ensemble_train_pred.reshape(-1, 1)).flatten()
ensemble_test_pred_orig = scaler_y.inverse_transform(ensemble_test_pred.reshape(-1, 1)).flatten()
y_test_final_orig = scaler_y.inverse_transform(y_test_final.reshape(-1, 1)).flatten()

# Calculate ensemble performance
ensemble_train_mse = mean_squared_error(y_train_orig, ensemble_train_pred_orig)
ensemble_test_mse = mean_squared_error(y_test_final_orig, ensemble_test_pred_orig)
ensemble_train_r2 = r2_score(y_train_orig, ensemble_train_pred_orig)
ensemble_test_r2 = r2_score(y_test_final_orig, ensemble_test_pred_orig)
ensemble_mae = mean_absolute_error(y_test_final_orig, ensemble_test_pred_orig)

print(f"\nDeep Ensemble Performance:")
print(f"Train MSE: {ensemble_train_mse:.2f}, R²: {ensemble_train_r2:.4f}")
print(f"Test MSE: {ensemble_test_mse:.2f}, R²: {ensemble_test_r2:.4f}")
print(f"Test MAE: {ensemble_mae:.2f}")
print(f"Mean Prediction Uncertainty: ±{np.mean(ensemble_uncertainty):.2f}")

# Individual model performance comparison
print(f"\nIndividual Model Performance (Test):")
print(f"Feedforward-style: MSE={ff_test_mse:.2f}, R²={ff_test_r2:.4f}")
print(f"LSTM-style: MSE={lstm_test_mse:.2f}, R²={lstm_test_r2:.4f}")
print(f"Attention-style: MSE={attention_test_mse:.2f}, R²={attention_test_r2:.4f}")
print(f"Ensemble: MSE={ensemble_test_mse:.2f}, R²={ensemble_test_r2:.4f}")

# Calculate improvement
best_individual_mse = min(ff_test_mse, lstm_test_mse, attention_test_mse)
improvement = (best_individual_mse - ensemble_test_mse) / best_individual_mse * 100

print(f"\nEnsemble Improvement:")
print(f"Improvement over best individual model: {improvement:.2f}%")

if improvement > 5:
    print(f"✓ Significant ensemble improvement achieved!")
elif improvement > 0:
    print(f"✓ Modest ensemble improvement achieved.")
else:
    print(f"⚠ No improvement over individual models.")

In [ ]:
# Create comprehensive visualization of deep ensemble results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Deep Ensemble Forecasting Analysis', fontsize=16, fontweight='bold')

# Plot 1: Model Performance Comparison
models = ['Feedforward-style', 'LSTM-style', 'Attention-style', 'Ensemble']
mse_values = [ff_test_mse, lstm_test_mse, attention_test_mse, ensemble_test_mse]
r2_values = [ff_test_r2, lstm_test_r2, attention_test_r2, ensemble_test_r2]

x = np.arange(len(models))
width = 0.35

bars1 = axes[0, 0].bar(x - width/2, mse_values, width, label='MSE', color='orange', alpha=0.7)
ax2 = axes[0, 0].twinx()
bars2 = ax2.bar(x + width/2, r2_values, width, label='R-squared', color='blue', alpha=0.7)

axes[0, 0].set_xlabel('Model')
axes[0, 0].set_ylabel('MSE', color='orange')
ax2.set_ylabel('R-squared', color='blue')
axes[0, 0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models, rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Ensemble Weights
colors = ['skyblue', 'lightcoral', 'lightgreen']
bars3 = axes[0, 1].bar(model_names, optimal_weights, color=colors, alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('Model')
axes[0, 1].set_ylabel('Ensemble Weight')
axes[0, 1].set_title('Learned Ensemble Weights', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Add value labels on bars
for bar, weight in zip(bars3, optimal_weights):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{weight:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Predictions vs Actual with Uncertainty
sample_indices = range(min(30, len(y_test_final_orig)))  # Show first 30 predictions
y_sample = y_test_final_orig[:len(sample_indices)]
pred_sample = ensemble_test_pred_orig[:len(sample_indices)]
uncertainty_sample = ensemble_uncertainty[:len(sample_indices)] * scaler_y.scale_[0]  # Convert uncertainty back

axes[1, 0].plot(sample_indices, y_sample, 'o-', label='Actual', linewidth=2, markersize=6, color='blue')
axes[1, 0].plot(sample_indices, pred_sample, 's--', label='Ensemble Prediction', linewidth=2, markersize=6, color='red')

# Add uncertainty bands
axes[1, 0].fill_between(sample_indices, 
                        pred_sample - uncertainty_sample, 
                        pred_sample + uncertainty_sample, 
                        alpha=0.3, color='red', label='Prediction Uncertainty')

axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Throughput')
axes[1, 0].set_title('Ensemble Predictions with Uncertainty', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Feature Importance from Attention Model
top_features = feature_importance[:5]
features = [f[0].replace('_', ' ').title() for f in top_features]
importances = [f[1] for f in top_features]

bars4 = axes[1, 1].barh(features, importances, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Importance Score')
axes[1, 1].set_title('Top 5 Feature Importance (Attention Model)', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Add value labels on bars
for bar, importance in zip(bars4, importances):
    width = bar.get_width()
    axes[1, 1].text(width + width*0.01, bar.get_y() + bar.get_height()/2.,
                    f'{importance:.3f}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Additional analysis: Feature importance from attention network
print("\n" + "=" * 60)
print("ATTENTION NETWORK FEATURE IMPORTANCE")
print("=" * 60)

print("\nTop 5 Most Important Features (according to attention mechanism):")
for i, (feature, importance) in enumerate(feature_importance[:5], 1):
    print(f"  {i}. {feature}: {importance:.4f}")

print("\nBottom 3 Least Important Features:")
for i, (feature, importance) in enumerate(feature_importance[-3:], len(feature_importance)-2):
    print(f"  {i}. {feature}: {importance:.4f}")

In [ ]:
# Risk analysis and practical implications
print("\n" + "=" * 60)
print("RISK ANALYSIS AND PRACTICAL IMPLICATIONS")
print("=" * 60)

# Calculate prediction intervals for risk assessment
confidence_level = 0.95
z_score = 1.96  # For 95% confidence

# Risk metrics
high_uncertainty_threshold = np.percentile(ensemble_uncertainty, 75)
high_uncertainty_indices = np.where(ensemble_uncertainty > high_uncertainty_threshold)[0]

print(f"\nUncertainty Analysis:")
print(f"  Mean prediction uncertainty: ±{np.mean(ensemble_uncertainty):.2f}")
print(f"  Uncertainty standard deviation: {np.std(ensemble_uncertainty):.2f}")
print(f"  High uncertainty threshold (75th percentile): ±{high_uncertainty_threshold:.2f}")
print(f"  High uncertainty predictions: {len(high_uncertainty_indices)}/{len(ensemble_uncertainty)} ({len(high_uncertainty_indices)/len(ensemble_uncertainty)*100:.1f}%)")

# Business risk assessment
port_capacity = 5000  # Example port capacity
capacity_utilization = y_test_final_orig / port_capacity * 100

high_utilization_risk = np.where(capacity_utilization > 85)[0]
print(f"\nCapacity Utilization Risk:")
print(f"  Predictions exceeding 85% capacity: {len(high_utilization_risk)}/{len(capacity_utilization)} ({len(high_utilization_risk)/len(capacity_utilization)*100:.1f}%)")
print(f"  Average capacity utilization: {np.mean(capacity_utilization):.1f}%")

# Decision support recommendations
print(f"\nDecision Support Recommendations:")
print(f"1. High Uncertainty Predictions:")
if len(high_uncertainty_indices) > 0:
    print(f"   - {len(high_uncertainty_indices)} predictions have high uncertainty")
    print(f"   - Consider additional data collection or expert judgment for these cases")
    print(f"   - Implement contingency planning for high-uncertainty scenarios")
else:
    print(f"   - All predictions have acceptable uncertainty levels")
    print(f"   - Current model reliability is sufficient for operational planning")

print(f"\n2. Capacity Management:")
if len(high_utilization_risk) > 0:
    print(f"   - {len(high_utilization_risk)} predictions indicate high capacity utilization")
    print(f"   - Consider proactive capacity expansion or demand management")
    print(f"   - Implement dynamic pricing or load balancing strategies")
else:
    print(f"   - Current capacity appears adequate for predicted demand")
    print(f"   - Maintain optimal resource allocation levels")

print(f"\n3. Model Reliability:")
print(f"   - Ensemble R²: {ensemble_test_r2:.4f} (Excellent if >0.8)")
print(f"   - Mean Absolute Error: {ensemble_mae:.2f} TEU")
if ensemble_test_r2 > 0.8:
    print(f"   - Model shows excellent predictive performance")
    print(f"   - Suitable for strategic planning and investment decisions")
elif ensemble_test_r2 > 0.6:
    print(f"   - Model shows good predictive performance")
    print(f"   - Suitable for operational planning with some caution")
else:
    print(f"   - Model performance needs improvement")
    print(f"   - Consider additional features or model refinement")

print(f"\n" + "=" * 60)
print("FINAL ENSEMBLE SUMMARY")
print("=" * 60)
print(f"Model Architecture: Deep Ensemble (Feedforward + LSTM + Attention styles)")
print(f"Optimal Weights: {dict(zip(model_names, [f'{w:.3f}' for w in optimal_weights]))}")
print(f"Test Performance: MSE={ensemble_test_mse:.2f}, R²={ensemble_test_r2:.4f}, MAE={ensemble_mae:.2f}")
print(f"Uncertainty Quantification: ±{np.mean(ensemble_uncertainty):.2f} TEU")
print(f"Key Features: {[f[0] for f in feature_importance[:3]]}")
print("=" * 60)

### Why this Tier exists vs Tier 3
Tier 4 addresses the **nonlinear pattern discovery** limitations of previous approaches:
- **Automatic feature learning** - discovers complex patterns without manual engineering
- **Multiple architecture types** - captures different aspects of relationships (temporal, attention, nonlinear)
- **Uncertainty quantification** - provides risk assessment for decision making
- **Ensemble robustness** - combines diverse models for better generalization

### Pros / Cons vs Tier 3
**Pros vs Tier 3:**
- **Nonlinear pattern discovery** - automatically learns complex relationships
- **Temporal modeling** - sequence-based models capture time dependencies
- **Feature importance** - attention-like mechanisms provide interpretability
- **Uncertainty quantification** - essential for risk-aware decision making
- **Ensemble robustness** - combines diverse models for better performance
- **Automatic feature learning** - no manual feature engineering required

**Cons vs Tier 3:**
- **Computational complexity** - much higher training and inference costs
- **Black box nature** - harder to interpret than linear models
- **Data requirements** - needs more data to train effectively
- **Hyperparameter sensitivity** - performance depends on architecture choices
- **Training instability** - complex models can be difficult to train reliably
- **Overfitting risk** - complex models may memorize training data

### When to use this Tier
- **Complex nonlinear relationships** where linear models fail
- **Temporal dependencies** in the data that need modeling
- **Large datasets** with sufficient samples for deep learning
- **Risk-critical applications** requiring uncertainty quantification
- **Pattern discovery** when relationships between variables are unknown
- **High-accuracy requirements** where model performance is paramount
- **Interpretability needs** through attention mechanisms and feature importance